# ST-OMR Meter V4-5 — One-Time Independent Final Holdout Evaluation

**Tek-seferlik final değerlendirme.** Bu notebook eğitim/tuning yapmaz. V4-4 ile dondurulan 150 örneği, V4-2 aday checkpoint'i ile yalnız bir kez değerlendirir.

Güvenlik sırası: **PRECHECK (checkpoint kapalı) → kalıcı ONE_SHOT_LOCK → checkpoint açma → tam 150 inference → immutable result.**

Final gate önceden donduruldu: accuracy ≥ 0.90, macro-F1 ≥ 0.90 ve 2/3/4 recall değerlerinin her biri ≥ 0.90. Sonucu gördükten sonra eşik değiştirilemez ve bu holdout ile tuning yapılamaz.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import shutil, subprocess, sys

EXPECTED_CODE_SHA = '3a20d7b37c31f65ebf5602378a07d02c25f36b89'
REPO_URL = 'https://github.com/khfy7wpr5p-maker/st-omr-training.git'
WORK_ROOT = Path('/content/st-omr-meter-v4-5')
REPO_DIR = WORK_ROOT / 'repo'
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(REPO_DIR)], check=True)
subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach',EXPECTED_CODE_SHA], check=True)
HEAD = subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'], text=True).strip()
assert HEAD == EXPECTED_CODE_SHA, (HEAD, EXPECTED_CODE_SHA)
print('V4-5 pinned code HEAD=', HEAD)


## Install exact pinned CPU runtime

Bu adım gerçek holdout'u veya checkpoint'i açmaz.


In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--extra-index-url', 'https://download.pytorch.org/whl/cpu',
    '-r', str(REPO_DIR/'requirements-training.txt')
], check=True)
runtime = subprocess.check_output([sys.executable,'-c','import torch; print(torch.__version__)'], text=True).strip()
print('PINNED TORCH=', runtime)
if runtime != '2.13.0+cpu':
    raise RuntimeError(f'Pinned Torch mismatch: {runtime}')
sys.path.insert(0, str(REPO_DIR))


In [ ]:
HOLDOUT_ROOT = Path('/content/drive/MyDrive/TEST/METER_V1/03_FINAL_HOLDOUT_150')
MANIFEST = HOLDOUT_ROOT / 'FINAL_HOLDOUT_150_SELECTION.json'
COMPLETION = HOLDOUT_ROOT / 'FINAL_HOLDOUT_150_BBOX_COMPLETE.json'
V42_ROOT = Path('/content/drive/MyDrive/TEST/METER_V1/02_ADAPTATION_RUNS/meter-v4-2-full-train-dev-screen-eb9c4b47c8fa')
V42_RESULT = V42_ROOT / 'result.json'
CHECKPOINT = V42_ROOT / 'numerator-specialist-development-candidate.pt'
HUMAN_EVIDENCE = REPO_DIR / 'METER_V4_4_HUMAN_VISUAL_REVIEW_EVIDENCE.json'
PREREG = REPO_DIR / 'METER_V4_5_PREREGISTRATION.json'
RUNS_ROOT = Path('/content/drive/MyDrive/TEST/METER_V1/02_ADAPTATION_RUNS')
OUTPUT = RUNS_ROOT / f'meter-v4-5-one-time-final-holdout-evaluation-{EXPECTED_CODE_SHA[:12]}'
LOCK = OUTPUT.with_name(OUTPUT.name + '.ONE_SHOT_LOCK.json')
PART = OUTPUT.with_name('.' + OUTPUT.name + '.part')
required = [HOLDOUT_ROOT, MANIFEST, COMPLETION, V42_RESULT, CHECKPOINT, HUMAN_EVIDENCE, PREREG, RUNS_ROOT]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required V4-5 input(s): ' + repr(missing))
print('OUTPUT=', OUTPUT)
print('LOCK=', LOCK)


## PRECHECK — checkpoint hâlâ kapalı

Bu hücre 150 görüntünün ve bbox'ların hash/geometry bağlarını kontrol eder ve numerator tensörlerini deterministik olarak hazırlar. **Checkpoint deserialize edilmez, inference yapılmaz.** Precheck FAIL ise final hücreye geçme.


In [ ]:
from st_omr_training.meter_v4_5_final_holdout_evaluation import (
    validate_preregistration, validate_human_review_evidence,
    prepare_final_holdout_v4_5, validate_v4_2_result,
    validate_checkpoint_file_hash,
)
if OUTPUT.exists() or PART.exists() or LOCK.exists():
    raise RuntimeError('V4-5 output/partial/lock already exists. Do NOT rerun the final evaluation.')
pre = validate_preregistration(PREREG)
human = validate_human_review_evidence(HUMAN_EVIDENCE)
prepared = prepare_final_holdout_v4_5(
    candidate_root=HOLDOUT_ROOT, manifest_path=MANIFEST,
    completion_receipt_path=COMPLETION,
)
v42 = validate_v4_2_result(V42_RESULT)
checkpoint = validate_checkpoint_file_hash(CHECKPOINT)
print('PRECHECK=PASS')
print('records=', len(prepared), 'human_review=', human['review_status'])
print('checkpoint_sha_verified=True')
print('CHECKPOINT_OPENED=False; MODEL_EVALUATED=False; INFERENCE_COUNT=0')
del prepared


## IRREVERSIBLE ONE-SHOT FINAL EVALUATION

Bu hücre çalıştığında, tüm preflight yeniden doğrulanır ve **checkpoint açılmadan hemen önce kalıcı ONE_SHOT_LOCK oluşturulur**. Lock oluştuğu andan sonra aynı final holdout için otomatik ikinci deneme yasaktır.

Yanlışlıkla “Tümünü çalıştır” kullanılırsa hücre burada bekler. Devam etmek için aşağıdaki cümleyi kutuya **aynen** yaz.


In [ ]:
ARM = input('Type exactly RUN_V4_5_FINAL_ONCE to open the frozen candidate once: ').strip()
if ARM != 'RUN_V4_5_FINAL_ONCE':
    raise RuntimeError('V4-5 final evaluation NOT armed; checkpoint remains closed.')
from st_omr_training.meter_v4_5_final_holdout_evaluation import run_meter_v4_5_one_time_final_holdout_evaluation
result = run_meter_v4_5_one_time_final_holdout_evaluation(
    candidate_root=HOLDOUT_ROOT,
    manifest_path=MANIFEST,
    completion_receipt_path=COMPLETION,
    human_review_evidence_path=HUMAN_EVIDENCE,
    preregistration_path=PREREG,
    v4_2_result_path=V42_RESULT,
    checkpoint_path=CHECKPOINT,
    output_root=OUTPUT,
    git_commit_sha=HEAD,
)
print('V4-5 FINAL EVALUATION COMPLETE')
print('decision=', result['decision'])
print('accuracy=', result['final_holdout']['accuracy'])
print('macro_f1=', result['final_holdout']['macro_f1'])
print('per_class_recall=', result['final_holdout']['per_class_recall'])
print('confusion=', result['final_holdout']['confusion'])
print('safety=', result['safety'])


## Read-only result view

Bu hücre mevcut sonucu yalnız okur. Modeli veya checkpoint'i yeniden açmaz.


In [ ]:
import hashlib, json
RESULT_PATH = OUTPUT / 'result.json'
COMPLETE_PATH = OUTPUT / 'COMPLETE'
if not RESULT_PATH.is_file() or not COMPLETE_PATH.is_file():
    raise RuntimeError('V4-5 immutable result is not complete')
raw = RESULT_PATH.read_bytes()
result_sha = hashlib.sha256(raw).hexdigest()
expected_complete = f'{result_sha}  result.json\n'.encode('ascii')
if COMPLETE_PATH.read_bytes() != expected_complete:
    raise RuntimeError('V4-5 COMPLETE binding mismatch')
view = json.loads(raw.decode('ascii'))
print('RESULT_SHA256=', result_sha)
print(json.dumps({
    'decision': view['decision'],
    'final_holdout': {k:v for k,v in view['final_holdout'].items() if k != 'predictions'},
    'safety': view['safety'],
}, indent=2, ensure_ascii=False))
